# 머신러닝 팀 프로젝트 안내서
## 데이터 분석과 머신러닝을 통한 은행 마케팅 전략 수립

- **운영:** 3인 1팀 / 09:00–18:00 (점심 13:00–14:00, 실제 8시간)
- **공통 목표:** 고객의 정기예금 가입 여부(`y`) 예측
- **자율 탐구:** 같은 데이터에서 팀별로 서로 다른 EDA 인사이트와 마케팅 전략 도출
- **제출물:** 완성한 `.ipynb`, 발표 자료(PPT/PDF)


## 데이터 안내
- 데이터: [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing)
- 권장 파일: `bank-full.csv` (45,211개 행, 입력 변수 16개, 타깃 `y`)
- CSV 구분자는 일반적인 쉼표가 아닌 **세미콜론(`;`)**입니다.
- 목표: `y`의 `yes` / `no` 이진 분류

**중요:** 고객에게 전화하기 *전에* 가입 가능성을 예측하는 프로젝트이므로 통화 종료 후에야 알 수 있는 `duration`은 모델 입력에서 제외하세요. `pdays=-1` 등 특수 코딩값의 의미도 확인하세요.


In [351]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO

url = "https://raw.githubusercontent.com/suyongzz/AICampus/refs/heads/main/bank-full.csv"

# 원본 파일이 세미콜론 구분 CSV이면 여기서 17개 컬럼으로 읽힙니다.
bank_full = pd.read_csv(url, sep=";", quotechar='"')

# 전체 행이 한 컬럼에 들어 있는 이중 CSV 형태라면 한 번 더 해석합니다.
if bank_full.shape[1] == 1:
    wrapped = pd.read_csv(url)
    text = "\n".join([wrapped.columns[0]] + wrapped.iloc[:, 0].astype(str).tolist())
    bank_full = pd.read_csv(StringIO(text), sep=";", quotechar='"')

assert bank_full.shape[1] == 17, f"예상 컬럼 수 17개, 실제 {bank_full.shape[1]}개"
assert 'y' in bank_full.columns

print("bank_full:", bank_full.shape)
print("컬럼명:", bank_full.columns.tolist())
display(bank_full.head())


bank_full: (45111, 17)
컬럼명: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 컬럼 이해하기

### 고객 기본 정보

| 컬럼 | 의미 |
|---|---|
| `age` | 고객 나이 |
| `job` | 직업 |
| `marital` | 결혼 상태 |
| `education` | 교육 수준 |
| `default` | 신용 불이행 여부 |
| `balance` | 평균 연간 잔액 |
| `housing` | 주택대출 여부 |
| `loan` | 개인대출 여부 |

### 이번 캠페인 정보

| 컬럼 | 의미 |
|---|---|
| `contact` | 연락 방식 |
| `day` | 마지막 연락일 |
| `month` | 마지막 연락 월 |
| `duration` | 마지막 통화시간(초) |
| `campaign` | 이번 캠페인 동안 연락한 횟수 |

### 이전 캠페인 정보

| 컬럼 | 의미 |
|---|---|
| `pdays` | 이전 캠페인 연락 후 지난 일수. `-1`은 이전 연락 없음 |
| `previous` | 이전 캠페인 연락 횟수 |
| `poutcome` | 이전 캠페인 결과 |

### Target

| 컬럼 | 의미 |
|---|---|
| `y` | 정기예금 가입 여부 (`yes` / `no`) |

### `duration` 주의
`duration`은 예측력이 높지만 **통화가 끝난 뒤에야 알 수 있는 값**입니다.

- 통화 종료 후 예측 성능 자체가 목적 → 사용 가능
- 전화하기 전 고객 선정이 목적 → 사용하면 안 됨

두 경우를 모두 비교합니다.

## 09:00–10:00 | 1단계: 개인별 데이터 탐색 및 EDA (60분)
**팀원 각자 독립적으로 분석합니다.**

- 데이터 구조, 타입, 결측·중복·특이값 점검
- 타깃 불균형 및 수치형·범주형 변수 분포 분석

In [352]:
print(bank_full.info())
print(bank_full.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 45111 entries, 0 to 45110
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45111 non-null  int64
 1   job        45111 non-null  str  
 2   marital    45111 non-null  str  
 3   education  45111 non-null  str  
 4   default    45111 non-null  str  
 5   balance    45111 non-null  int64
 6   housing    45111 non-null  str  
 7   loan       45111 non-null  str  
 8   contact    45111 non-null  str  
 9   day        45111 non-null  int64
 10  month      45111 non-null  str  
 11  duration   45111 non-null  int64
 12  campaign   45111 non-null  int64
 13  pdays      45111 non-null  int64
 14  previous   45111 non-null  int64
 15  poutcome   45111 non-null  str  
 16  y          45111 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB
None
0


In [353]:
# 범주형 컬럼 'unknown' : job / education / contact / poutcome     

cat_cols = bank_full.select_dtypes(exclude='number').columns
print(bank_full[cat_cols].eq('unknown').sum())

# 수치형 컬럼 기초 통계량
display(bank_full.describe().T)

# 특정 컬럼의 특수값 확인
print(bank_full['pdays'].value_counts().head(10))

# job 클래스별 비율 (%)
print(bank_full['job'].value_counts(normalize=True) * 100)  


job            286
marital          0
education     1852
default          0
housing          0
loan             0
contact      13017
month            0
poutcome     36932
y                0
dtype: int64


,count,mean,std,min,25%,50%,75%,max
age,45111.0,40.928000,10.603352,18.0,33.0,39.0,48.0,95.0
balance,45111.0,1361.723837,3046.001767,-8019.0,72.0,447.0,1425.0,102127.0
day,45111.0,15.806854,8.325074,1.0,8.0,16.0,21.0,31.0
duration,45111.0,258.042695,257.574409,0.0,103.0,180.0,318.0,4918.0
campaign,45111.0,2.766443,3.100673,1.0,1.0,2.0,3.0,63.0
pdays,45111.0,39.978098,99.787561,-1.0,-1.0,-1.0,-1.0,854.0
previous,45111.0,0.574184,2.296670,0.0,0.0,0.0,0.0,275.0


pdays
-1      36927
 182      167
 92       137
 183      124
 91       119
 181      117
 370       99
 184       83
 364       77
 350       73
Name: count, dtype: int64
job
blue-collar      21.553501
management       20.928377
technician       16.802997
admin.           11.434018
services          9.195097
retired           4.983264
self-employed     3.495821
entrepreneur      3.285230
unemployed        2.881780
housemaid         2.746558
student           2.059365
unknown           0.633992
Name: proportion, dtype: float64


In [354]:
from sklearn.model_selection import train_test_split

# drop_li = ['y', 'duration', 'poutcome']
drop_li = ['y', 'duration']

X = bank_full.drop(columns=drop_li)
X['HasPreviousContact'] = (X['pdays'] != -1).astype(int)
y = bank_full['y']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.15, random_state=42)

In [355]:
# 학습 데이터 복사
X_tr = X_tr.copy()
X_te = X_te.copy()

# job, education의 unknown을 최빈값으로 변경
fill_modes = {}

for col in ['job', 'education']:

    # unknown을 제외한 최빈값 계산
    mode = X_tr.loc[
        X_tr[col] != 'unknown', col
    ].mode()[0]

    fill_modes[col] = mode

    # 학습 데이터에 적용
    X_tr[col] = X_tr[col].replace('unknown', mode)

    # 테스트 데이터에 동일한 값 적용
    X_te[col] = X_te[col].replace('unknown', mode)

# 처리 결과 확인
print(fill_modes)
print(X_tr[['job', 'education', 'contact']].eq('unknown').sum())

{'job': 'blue-collar', 'education': 'secondary'}
job              0
education        0
contact      11063
dtype: int64


In [356]:
X_tr = pd.get_dummies(X_tr, dtype=int)
X_te = pd.get_dummies(X_te, dtype=int)

# 두 데이터의 열을 동일하게 맞춤
X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

In [357]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

model_lr = LogisticRegression(max_iter=1000)

model_rf = RandomForestClassifier(random_state=42, n_jobs=-1)

In [358]:
from sklearn.metrics import accuracy_score

model_lr.fit(X_tr, y_tr)
y_pred_lr = model_lr.predict(X_te)
print("lr :",accuracy_score(y_te, y_pred_lr))

model_rf.fit(X_tr, y_tr)
y_pred_rf = model_rf.predict(X_te)
print("rf :",accuracy_score(y_te, y_pred_rf))

c:\Users\user\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


lr : 0.889759125166248
rf : 0.8870991576769617


In [359]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)

threshold = 0.4

# yes 클래스 위치 확인
yes_idx = list(model_rf.classes_).index('yes')

# 가입 예측 확률
yes_proba = model_rf.predict_proba(X_te)[:, yes_idx]

# Threshold 0.4 적용
y_pred = np.where(yes_proba >= threshold, 'yes', 'no')

# 성능 평가
print("Accuracy:", accuracy_score(y_te, y_pred))
print(classification_report(y_te, y_pred))
print(confusion_matrix(y_te, y_pred, labels=['no', 'yes']))

# 예측 결과 확인
result = pd.DataFrame({
    '실제값': y_te,
    '예측값': y_pred,
    '가입확률': yes_proba.round(3)
}, index=X_te.index)

display(result.head(10))

Accuracy: 0.8829614304714053
              precision    recall  f1-score   support

          no       0.91      0.96      0.94      5955
         yes       0.52      0.32      0.40       812

    accuracy                           0.88      6767
   macro avg       0.72      0.64      0.67      6767
weighted avg       0.86      0.88      0.87      6767

[[5715  240]
 [ 552  260]]


,실제값,예측값,가입확률
23200,no,no,0.01
14631,no,no,0.07
39686,yes,no,0.15
44025,no,no,0.28
18575,no,no,0.15
21749,no,no,0.03
38190,no,no,0.28
10048,no,no,0.02
41617,yes,yes,0.72
2644,no,no,0.00


In [360]:
from sklearn.metrics import classification_report, confusion_matrix

print("Logistic Regression")
print(classification_report(y_te, y_pred_lr))

print("Random Forest")
print(classification_report(y_te, y_pred_rf))

print(confusion_matrix(
    y_te, y_pred_rf,
    labels=['no', 'yes']
))

Logistic Regression
              precision    recall  f1-score   support

          no       0.90      0.99      0.94      5955
         yes       0.67      0.16      0.26       812

    accuracy                           0.89      6767
   macro avg       0.78      0.58      0.60      6767
weighted avg       0.87      0.89      0.86      6767

Random Forest
              precision    recall  f1-score   support

          no       0.90      0.98      0.94      5955
         yes       0.58      0.22      0.32       812

    accuracy                           0.89      6767
   macro avg       0.74      0.60      0.63      6767
weighted avg       0.86      0.89      0.86      6767

[[5822  133]
 [ 631  181]]


In [370]:
import numpy as np
import pandas as pd

# 1. 새로운 고객 데이터 불러오기
bank_submission = pd.read_csv("input_bank_submission.csv")

# 2. 학습 데이터와 동일한 특성 공학
bank_submission['HasPreviousContact'] = (
    bank_submission['pdays'] != -1
).astype(int)

# 3. 학습할 때 사용한 컬럼 선택
X_submit = bank_submission[X.columns].copy()

# 4. job, education의 unknown을 최빈값으로 대체
for col, mode in fill_modes.items():
    X_submit[col] = X_submit[col].replace('unknown', mode)

# 5. One-Hot Encoding
X_submit = pd.get_dummies(X_submit, dtype=int)

# 6. 학습 데이터와 컬럼 일치
X_submit = X_submit.reindex(
    columns=X_tr.columns,
    fill_value=0
)

# 7. Logistic Regression 예측
pred_lr = model_lr.predict(X_submit)

# 8. RandomForest 가입 확률 예측
yes_idx = list(model_rf.classes_).index('yes')
rf_proba = model_rf.predict_proba(X_submit)[:, yes_idx]

# Threshold = 0.4 적용
pred_rf = np.where(rf_proba >= 0.4, 'yes', 'no')

# 9. 예측 결과 확인
result = pd.DataFrame({
    'LR_가입예측': pred_lr,
    'RF_가입예측': pred_rf,
    'RF_가입확률': rf_proba.round(3)
})

display(result.head(10))

# 10. RandomForest 제출 결과 저장
result_rf = pd.DataFrame({
    'y': pred_rf
})

result_rf.to_csv("bank_predictions.csv", index=False)

print("예측 결과 저장 완료")

,LR_가입예측,RF_가입예측,RF_가입확률
0,no,yes,0.48
1,yes,yes,0.69
2,yes,yes,0.59
3,no,yes,0.45
4,no,no,0.22
5,yes,yes,0.47
6,no,yes,0.49
7,yes,yes,0.57
8,no,no,0.14
9,no,yes,0.64


예측 결과 저장 완료


In [367]:
# 학습 데이터에 해당하는 원본 행만 사용
eda_df = bank_full.loc[X_tr.index].copy()

for col in ['poutcome', 'contact', 'education', 'job']:

    temp = eda_df.groupby(col)['y'].agg(
        고객수='size',
        가입자수=lambda x: (x == 'yes').sum()
    )

    temp['가입률(%)'] = (
        temp['가입자수'] / temp['고객수'] * 100
    ).round(1)

    print(f"\n[{col}별 가입률]")
    display(temp.sort_values('가입률(%)', ascending=False))


[poutcome별 가입률]


,고객수,가입자수,가입률(%)
poutcome,,,
success,1248,814,65.2
other,1566,263,16.8
failure,4127,521,12.6
unknown,31403,2831,9.0



[contact별 가입률]


,고객수,가입자수,가입률(%)
contact,,,
cellular,24823,3669,14.8
telephone,2458,330,13.4
unknown,11063,430,3.9



[education별 가입률]


,고객수,가입자수,가입률(%)
education,,,
tertiary,11253,1683,15.0
unknown,1582,209,13.2
secondary,19715,2053,10.4
primary,5794,484,8.4



[job별 가입률]


,고객수,가입자수,가입률(%)
job,,,
student,786,222,28.2
retired,1932,425,22.0
unemployed,1085,158,14.6
management,8063,1104,13.7
admin.,4374,536,12.3
unknown,238,29,12.2
self-employed,1329,155,11.7
technician,6450,720,11.2
services,3521,312,8.9


In [368]:
# 테스트 데이터에 해당하는 원본 정보
error_df = bank_full.loc[
    X_te.index,
    ['job', 'contact', 'poutcome', 'previous', 'pdays', 'y']
].copy()

# 예측 결과 연결
error_df['예측'] = y_pred
error_df['가입확률'] = yes_proba

# 실제로 가입한 고객만 선택
actual_yes = error_df[error_df['y'] == 'yes'].copy()

# 놓친 고객: 실제 yes, 예측 no
actual_yes['놓침'] = (
    actual_yes['예측'] == 'no'
).astype(int)

# 그룹별 놓친 비율 분석
for col in ['poutcome', 'contact', 'job']:

    temp = actual_yes.groupby(col)['놓침'].agg(
        실제가입자='count',
        놓친가입자='sum',
        놓친비율='mean'
    )

    temp['놓친비율'] = (temp['놓친비율'] * 100).round(1)

    # 실제 가입자가 20명 이상인 그룹만 확인
    temp = temp[temp['실제가입자'] >= 20]

    print(f"\n[{col}별 미탐지 분석]")
    display(temp.sort_values('놓친비율', ascending=False))


[poutcome별 미탐지 분석]


,실제가입자,놓친가입자,놓친비율
poutcome,,,
failure,91,77,84.6
unknown,539,443,82.2
other,41,27,65.9
success,141,5,3.5



[contact별 미탐지 분석]


,실제가입자,놓친가입자,놓친비율
contact,,,
unknown,100,99,99.0
cellular,654,418,63.9
telephone,58,35,60.3



[job별 미탐지 분석]


,실제가입자,놓친가입자,놓친비율
job,,,
blue-collar,124,106,85.5
services,55,44,80.0
admin.,89,64,71.9
self-employed,31,22,71.0
technician,110,78,70.9
housemaid,23,15,65.2
unemployed,43,28,65.1
management,188,113,60.1
student,42,24,57.1


## 10:00–11:00 | 2단계: 인사이트 공유 및 문제 정의 (60분)
- A, B, C 각자 **10분씩** 분석 결과 설명
- 공통점과 차이점 토론, 중요한 분석 질문 선정
- **팀별 핵심 분석 주제 1개 이상**, 가설 **2개 이상** 작성

**우리 팀의 핵심 질문:**

**가설 1:**

**가설 2:**

**가설 3:**

**이 가설을 검증할 변수·그래프:**


## 11:00–12:30 | 3단계: 데이터 전처리 및 시각화 (90분)
- 결측·중복·이상치 검토 및 처리 근거 기록
- 범주형 변수의 One-Hot Encoding 또는 적절한 인코딩
- 스케일링이 필요한 모델 고려
- 팀별 가설 검증 시각화 **최소 3개** 작성
- `duration` 제외 여부와 그 이유 기록

**주의:** EDA는 전체 훈련용 자료를 이해하는 과정이고, 모델 평가에서는 테스트 정보가 전처리 학습에 섞이지 않게 해야 합니다. 분할 후 학습 데이터에만 `fit`하는 `Pipeline` / `ColumnTransformer`를 권장합니다.

### 전처리 결정 및 이유
-

### 핵심 그래프 해석 (최소 3개)
1.

2.

3.


## 12:30–13:00 | 4단계: 모델 선정 및 실험 설계 (30분)
- 공통 타깃: 예금 가입 여부 `y`
- **최소 3개 모델** (예: Logistic Regression, KNN, Random Forest, Gradient Boosting, Naive Bayes)
- 동일한 데이터 분할·평가 지표를 사용해 비교
- 분류 문제에서 Accuracy 외 Precision, Recall, F1 중 필요한 지표 선택
- **검증용 데이터 또는 교차검증**을 활용해 튜닝하고, 테스트셋은 맨 마지막에만 평가

| 모델 | 선정 이유 | 전처리 요구 | 평가 지표 |
|---|---|---|---|
| 모델 1 |  |  |  |
| 모델 2 |  |  |  |
| 모델 3 |  |  |  |


## 13:00–14:00 | 점심시간 (60분)


## 14:00–15:30 | 5단계: 머신러닝 모델 학습 및 성능 비교 (90분)
- 3개 이상 모델 학습 및 동일 조건에서 검증 성능 비교
- Confusion Matrix 등으로 모델별 오분류 양상 확인
- 단순 최고 정확도뿐 아니라 **은행 마케팅 목표에 적합한 평가 지표**를 선택

| 모델 | Train 점수 | Validation 점수 | 주요 특징/한계 |
|---|---:|---:|---|
| 모델 1 |  |  |  |
| 모델 2 |  |  |  |
| 모델 3 |  |  |  |

**어떤 모델이 가장 좋았으며 왜 그랬을까?**


## 15:30–16:15 | 6단계: 모델 개선 및 최종 평가 (45분)
- KNN 등에서 스케일링 전후 비교 또는 하이퍼파라미터 실험 **1개 이상**
- 학습·검증 성능 차이를 보고 과적합 가능성 평가
- 검증 결과를 기준으로 최종 모델 결정 후 **테스트셋 1회 최종 평가**
- 과적합이 없을 수도 있습니다. 관찰된 사실대로 기록하세요.

| 개선 실험 | 변경 전 검증 점수 | 변경 후 검증 점수 | 해석 |
|---|---:|---:|---|
|  |  |  |  |

**과적합 여부 및 근거:**

**최종 선택 모델 및 이유:**

**최종 테스트 성능:**


## 16:15–17:00 | 7단계: 결론 및 발표 자료 준비 (45분)
- 주요 EDA 인사이트 3개 요약
- 모델 3개 이상 비교와 개선 실험 설명
- 실제 은행에 제안할 **마케팅 전략 2개 이상** 작성
- 분석의 한계와 추가 데이터 필요성 기록
- 발표 자료 구성: 목표 → 데이터 → EDA → 전처리 → 모델 비교 → 개선 → 전략 → 한계

**전략 1:**

**전략 2:**

**관찰 자료만으로 인과 효과를 확정할 수 없는 점 등 한계:**


## 17:00–18:00 | 8단계: 팀별 발표 및 피드백 (60분)
**3팀 기준:** 각 팀 발표 10분 + 질의응답 5분, 강사 종합 피드백 15분

- 두 팀원 모두 발표 참여
- 핵심 EDA, 모델 비교 및 개선, 모델 선정 이유, 마케팅 전략을 명확히 제시


## 최종 제출 체크리스트
- [ ] 팀원별 EDA 코드 및 인사이트 3개씩
- [ ] 팀별 분석 주제 1개, 가설 2개 이상
- [ ] 팀별 핵심 시각화 그래프 3개 이상
- [ ] 전처리 방법과 Data Leakage 방지 근거
- [ ] 머신러닝 모델 3개 이상 비교
- [ ] 스케일링 또는 하이퍼파라미터 개선 실험 1회 이상
- [ ] 최종 테스트 평가 결과와 모델 선정 이유
- [ ] `.ipynb`와 발표 자료 제출



---
## 핵심 분석 주제 TOP 5 — 참고 예시
**아래는 서로 다른 예측 타깃이 아닙니다.** 모든 팀의 머신러닝 공통 목표는 `y`(예금 가입 여부) 예측이며, 아래 주제는 EDA와 마케팅 전략을 발전시키기 위한 **선택형 탐구 방향**입니다.

### TOP 1. 고객 특성에 따른 가입률 분석
- 연령, 직업, 교육 수준, 잔액, 대출 보유 여부별 가입률 비교
- 질문: *어떤 고객 특성이 예금 가입과 관련되는가?*

### TOP 2. 마케팅 연락 횟수와 가입률의 관계
- `campaign`, `previous`, `poutcome` 등의 관계 탐색
- 질문: *연락 횟수 및 과거 캠페인 경험에 따라 가입률은 달라지는가?*
- 관찰된 관계를 추가 연락의 인과 효과로 단정하지 않기

### TOP 3. 고객 세분화와 맞춤형 전략
- 나이·대출·계좌 잔액 등으로 그룹 정의, 그룹별 가입률 비교
- 질문: *어떤 고객군에서 서로 다른 특성이 관찰되는가?*

### TOP 4. 캠페인 시기와 연락 방식 분석
- `month`, `contact`에 따른 가입률 및 고객 구성 비교
- 질문: *시기·연락 방식에 따라 어떤 차이가 나타나는가?*
- 월·방식의 차이를 곧바로 인과 효과로 해석하지 않기

### TOP 5. 머신러닝 기반 우선 연락 대상 선정
- 예측 확률 상위 고객과 무작위 선정 고객의 실제 가입률 비교
- 질문: *같은 연락 예산에서 어떤 모델이 가입 고객을 더 잘 찾아내는가?*

**마지막 질문:** 모델의 정확도뿐 아니라 분석 근거, 비즈니스 목표, 데이터 누수와 일반화 문제를 고려하여 왜 그 모델을 최종 선택했는지 설명해 보세요.
